In [ ]:
# Parameters
software='vlc'

In [ ]:
# ================= DeepOD base_networks — pipeline completo e corrigido =================
# Requisitos: deepod >= 0.4.x, scikit-learn, torch
import os, random, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
#from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, accuracy_score
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else torch.device("cpu"))
print(f"[Runtime] seed={seed} device={device.type}")

import numpy as np, os, torch
from sklearn.preprocessing import StandardScaler


In [ ]:
import numpy as np
import os, torch
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

# ============================================================
# Project paths - compatible with standalone execution and batch execution
# Folder layout expected:
# CNSM_2026/
# ├── dado_pickle/
# ├── notebooks/
# ├── results/
# └── shared_artifacts/<software>/
# ============================================================
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", r"C:\CNSM_2026"))
NOTEBOOKS_DIR = Path(globals().get("NOTEBOOKS_DIR", PROJECT_ROOT / "notebooks"))
RESULTS_ROOT = Path(globals().get("RESULTS_ROOT", PROJECT_ROOT / "results"))
SHARED_ARTIFACTS_ROOT = Path(globals().get("SHARED_ARTIFACTS_ROOT", PROJECT_ROOT / "shared_artifacts"))
DADO_PICKLE_ROOT = Path(globals().get("DADO_PICKLE_ROOT", PROJECT_ROOT / "dado_pickle"))

software = str(globals().get("software", "core.exe"))
software_dir = SHARED_ARTIFACTS_ROOT / software
indices_path = software_dir / "indices_pad256_seed42.npz"
data_path = software_dir / "X_hex_y_pad256.npz"

if not software_dir.exists():
    raise FileNotFoundError(
        f"Software artifact folder not found: {software_dir}\n"
        f"Expected folder: {SHARED_ARTIFACTS_ROOT}/<software>/"
    )
if not indices_path.exists():
    raise FileNotFoundError(f"Split indices file not found: {indices_path}")
if not data_path.exists():
    raise FileNotFoundError(f"Processed dataset file not found: {data_path}")

print(f"[Paths] PROJECT_ROOT={PROJECT_ROOT}")
print(f"[Paths] SHARED_ARTIFACTS_ROOT={SHARED_ARTIFACTS_ROOT}")
print(f"[Data] software={software}")

# 1) Load the same split indices for every notebook/model
idx = np.load(indices_path, allow_pickle=True)
idx_train, idx_val, idx_test = idx["train"], idx["val"], idx["test"]

# 2) Load X and y produced by Stage 0
bundle = np.load(data_path, allow_pickle=True)
X_all = bundle["X_hex"]
y_all = bundle["y"].astype(np.int64)

# 3) Flatten for tabular/network style models and normalize without leakage
X_all_2d = X_all.reshape(X_all.shape[0], -1).astype(np.int64)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_all_2d[idx_train])
X_val   = scaler.transform(X_all_2d[idx_val])
X_test  = scaler.transform(X_all_2d[idx_test])

y_train, y_val, y_test = y_all[idx_train], y_all[idx_val], y_all[idx_test]

print("Tabular/Network shapes:", X_train.shape, X_val.shape, X_test.shape)
print("DNN:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:

import os, time, math, tempfile, psutil
import numpy as np
import torch
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report,
                             average_precision_score)
import torch.nn as nn
import matplotlib.pyplot as plt

# ===== Dados (iguais aos seus) =====
## Preparar dados para BLSTM: [B, T, 1]
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
y_val   = torch.tensor(y_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)


# X_train, y_train, X_val, y_val, X_test, y_test já criados acima
train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
val_dataset   = torch.utils.data.TensorDataset(X_val,   y_val)
test_dataset  = torch.utils.data.TensorDataset(X_test,  y_test)
batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)


In [ ]:
# Requisitos extras:
# %pip install psutil scikit-learn

import os, time, math, tempfile, psutil
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report,
                             average_precision_score)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ===== Modelo =====
class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, 1, bias=False)
    def forward(self, x):
        a = torch.tanh(self.proj(x))
        s = self.v(a).squeeze(-1)
        w = torch.softmax(s, dim=1)
        ctx = torch.bmm(w.unsqueeze(1), x).squeeze(1)
        return ctx, w

class BGRUDetector(nn.Module):
    def __init__(self, hidden=64, layers=1, dropout=0.3, conv_channels=8, ksize=5):
        super().__init__()
        self.conv = nn.Conv1d(1, conv_channels, kernel_size=ksize, padding=ksize//2)
        self.bn   = nn.BatchNorm1d(conv_channels)
        self.act  = nn.GELU()
        self.gru  = nn.GRU(input_size=conv_channels, hidden_size=hidden,
                           num_layers=layers, batch_first=True, bidirectional=True,
                           dropout=dropout if layers > 1 else 0.0)
        self.attn = TemporalAttention(2*hidden)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(2*hidden, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )
    def forward(self, x):
        x = x.unsqueeze(-1) if x.dim() == 2 else x    # [B,T] -> [B,T,1]
        x = x.transpose(1, 2)                         # [B,1,T]
        x = self.act(self.bn(self.conv(x)))           # [B,C,T]
        x = x.transpose(1, 2)                         # [B,T,C]
        out, _ = self.gru(x)                          # [B,T,2H]
        ctx, _ = self.attn(out)                       # [B,2H]
        return self.head(ctx)                         # [B,1] (logits)

# ===== Sistema e modelo =====
def system_snapshot():
    vm = psutil.virtual_memory()
    return dict(cpu_pct=psutil.cpu_percent(interval=None),
                ram_used_GB=(vm.total-vm.available)/(1024**3),
                ram_total_GB=vm.total/(1024**3))

def gpu_peak_reset():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()

def gpu_peak_mb():
    return torch.cuda.max_memory_allocated()/(1024**2) if torch.cuda.is_available() else float("nan")

def model_size_and_params(model):
    n_params = sum(p.numel() for p in model.parameters())
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".pt"); tmp.close()
    torch.save(model.state_dict(), tmp.name)
    size_mb = os.path.getsize(tmp.name)/(1024**2); os.unlink(tmp.name)
    return n_params, size_mb

# ===== Config =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BGRUDetector(hidden=64, layers=1, dropout=0.3, conv_channels=8, ksize=5).to(device)

pos = float((torch.tensor(y_train) == 1).sum().item())
neg = float((torch.tensor(y_train) == 0).sum().item())
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 100
best_val_auc = -np.inf
best_state = None
patience, bad = 10, 0

# ===== Logs =====
tr_loss_hist, va_loss_hist = [], []
tr_acc_hist,  va_acc_hist  = [], []
tr_auc_hist,  va_auc_hist  = [], []
tr_pra_hist,  va_pra_hist  = [], []
thr_tr_hist,  thr_va_hist  = [], []
cpu_hist, ram_hist, gpu_peak_hist = [], [], []

# ===== Época única (treino/val) com throughput, F1 e PR-AUC =====
def run_epoch(dl, train=False):
    model.train(train)
    gpu_peak_reset()
    t0 = time.time()
    total_loss, n = 0.0, 0
    all_logits, all_labels = [], []
    for xb, yb in dl:
        xb = xb.to(device).float()
        yb = yb.to(device).view(-1, 1).float()
        if train: optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        bs = xb.size(0)
        total_loss += loss.item() * bs
        n += bs
        all_logits.append(logits.detach().cpu())
        all_labels.append(yb.detach().cpu())
    dt = max(time.time() - t0, 1e-9)
    thr = n / dt

    avg_loss = total_loss / max(n, 1)
    if n:
        probs = torch.sigmoid(torch.cat(all_logits, 0)).numpy().ravel()
        labs  = torch.cat(all_labels, 0).numpy().ravel().astype(int)
        preds = (probs >= 0.5).astype(int)
        acc = accuracy_score(labs, preds)
        try:
            auc = roc_auc_score(labs, probs)
        except ValueError:
            auc = float("nan")
        try:
            pr_auc = average_precision_score(labs, probs)
        except ValueError:
            pr_auc = float("nan")
    else:
        acc = auc = pr_auc = float("nan")

    sys_s = system_snapshot()
    gpu_pk = gpu_peak_mb()
    return avg_loss, acc, auc, pr_auc, thr, sys_s, gpu_pk

# ===== Loop de treino/val =====
for epoch in range(1, epochs + 1):
    tr_loss, tr_acc, tr_auc, tr_pra, tr_thr, sys_tr, gpu_tr = run_epoch(train_loader, train=True)
    va_loss, va_acc, va_auc, va_pra, va_thr, sys_va, gpu_va = run_epoch(val_loader,   train=False)

    tr_loss_hist.append(tr_loss); va_loss_hist.append(va_loss)
    tr_acc_hist.append(tr_acc);   va_acc_hist.append(va_acc)
    tr_auc_hist.append(tr_auc);   va_auc_hist.append(va_auc)
    tr_pra_hist.append(tr_pra);   va_pra_hist.append(va_pra)
    thr_tr_hist.append(tr_thr);   thr_va_hist.append(va_thr)

    cpu_hist.append(sys_va["cpu_pct"]); ram_hist.append(sys_va["ram_used_GB"]); gpu_peak_hist.append(gpu_va)

    if va_auc > best_val_auc:
        best_val_auc = va_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    print(f"Epoch {epoch:03d} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} auc {tr_auc:.4f} prAUC {tr_pra:.4f} thr {tr_thr:.1f} | "
          f"val   loss {va_loss:.4f} acc {va_acc:.4f} auc {va_auc:.4f} prAUC {va_pra:.4f} thr {va_thr:.1f} | "
          f"CPU {sys_va['cpu_pct']:.1f}% RAM {sys_va['ram_used_GB']:.2f}GB GPU_peak {gpu_va:.1f}MB")

    if bad >= patience:
        print("Early stopping."); break

if best_state is not None:
    model.load_state_dict(best_state)
model.to(device)

# ===== Tamanho do modelo e nº de parâmetros =====
n_params, size_mb = model_size_and_params(model)
print(f"Parâmetros: {n_params} | Tamanho do modelo: {size_mb:.3f} MB")

# ===== Teste com throughput, F1 e PR-AUC =====
model.eval()
t0 = time.time(); 
gpu_peak_reset()
with torch.no_grad():
    all_probs, all_preds, all_labels = [], [], []
    test_loss, n_test = 0.0, 0
    for xb, yb in test_loader:
        xb = xb.to(device).float(); yb = yb.to(device).view(-1,1).float()
        logits = model(xb)
        loss = criterion(logits, yb)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        preds = (probs >= 0.5).astype(int)
        all_probs.append(probs); all_preds.append(preds); all_labels.append(yb.cpu().numpy().ravel().astype(int))
        test_loss += loss.item() * xb.size(0); n_test += xb.size(0)
dt = max(time.time()-t0, 1e-9)
test_thr = n_test / dt

proba = np.concatenate(all_probs); pred = np.concatenate(all_preds); ytrue = np.concatenate(all_labels)
acc  = accuracy_score(ytrue, pred)
prec, rec, f1, _ = precision_recall_fscore_support(ytrue, pred, average="binary", zero_division=0)
try:
    auc = roc_auc_score(ytrue, proba)
except ValueError:
    auc = float("nan")
pr_auc_test = average_precision_score(ytrue, proba)
cm = confusion_matrix(ytrue, pred)
test_loss = test_loss / max(n_test, 1)

print("\nResultados no teste")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")
print(f"PR-AUC:    {pr_auc_test:.4f}")
print(f"Throughput:{test_thr:.1f} samp/s")
print(f"Test loss: {test_loss:.4f}")
print("\nMatriz de confusão:\n", cm)
print("\nRelatório de classificação:")
print(classification_report(ytrue, pred, digits=4))

# ===== Gráficos por época =====
epochs_r = range(1, len(tr_loss_hist)+1)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.plot(epochs_r, tr_loss_hist, label='Train Loss'); plt.plot(epochs_r, va_loss_hist, label='Val Loss')
plt.xlabel('Época'); plt.ylabel('Loss'); plt.legend()
plt.subplot(1,2,2); plt.plot(epochs_r, tr_acc_hist, label='Train Acc'); plt.plot(epochs_r, va_acc_hist, label='Val Acc')
plt.xlabel('Época'); plt.ylabel('Acc'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.plot(epochs_r, va_auc_hist, label='ROC-AUC Val'); plt.plot(epochs_r, va_pra_hist, label='PR-AUC Val')
plt.xlabel('Época'); plt.ylabel('Score'); plt.legend()
plt.subplot(1,2,2); plt.plot(epochs_r, thr_tr_hist, label='Thr Train'); plt.plot(epochs_r, thr_va_hist, label='Thr Val')
plt.xlabel('Época'); plt.ylabel('samples/s'); plt.legend(); plt.tight_layout(); plt.show()

plt.figure(figsize=(12,4))
plt.plot(epochs_r, cpu_hist, label='CPU %'); plt.plot(epochs_r, ram_hist, label='RAM used (GB)')
if any(np.isfinite(gpu_peak_hist)): plt.plot(epochs_r, gpu_peak_hist, label='GPU peak (MB)')
plt.xlabel('Época'); plt.ylabel('Uso'); plt.legend(); plt.tight_layout(); plt.show()


## Standardized scientific result tracking

This final cell saves the notebook as an individual experiment run using the shared result schema. It creates tables, figures, metadata, checkpoint, and updates `results/results_summary_model.csv`.

Batch/dataset-ready update: this notebook accepts `RESULTS_ROOT`, `BATCH_ID`, `EXECUTION_NUMBER`, `RUN_ID`, `RUN_DIR`, `DATASET_ID`, `DATASET_MANIFEST_PATH`, `PROCESSING_RUN_DIR`, and `PROCESSED_DATA_ROOT` injected by the batch runner.

In [ ]:
# ============================================================
# Scientific result tracking - standardized output block
# Uses the evaluation already computed by this notebook.
# It does NOT run the model again on test_loader.
# ============================================================

STANDARD_TRACKING_VERSION = "v1.3-existing-evaluation-only"
NOTEBOOK_NAME = str(globals().get("NOTEBOOK_NAME", "unknown_notebook.ipynb"))

import os
import json
import math
import uuid
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)


def _json_safe(value):
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        if value.size > 5000:
            return f"<ndarray shape={value.shape} dtype={value.dtype}>"
        return _json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return None if not math.isfinite(value) else value
    if isinstance(value, torch.Tensor):
        arr = value.detach().cpu().numpy()
        return _json_safe(arr)
    if isinstance(value, Path):
        return str(value)
    return value


def _is_number(value):
    if isinstance(value, (int, float, np.integer, np.floating)):
        try:
            return math.isfinite(float(value))
        except Exception:
            return False
    return False


def _to_float(value, default=float("nan")):
    if _is_number(value):
        return float(value)
    return default


def _find_model():
    preferred_names = ["model", "gcn", "gcn_gru"]
    for name in preferred_names:
        obj = globals().get(name)
        if isinstance(obj, nn.Module):
            return name, obj
    for name, obj in globals().items():
        if isinstance(obj, nn.Module):
            return name, obj
    return None, None


def _get_device(model_obj):
    if model_obj is not None:
        try:
            return next(model_obj.parameters()).device
        except StopIteration:
            pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _as_array(value, *, score=False):
    if value is None:
        return None
    try:
        if isinstance(value, torch.Tensor):
            arr = value.detach().cpu().numpy()
        elif isinstance(value, (list, tuple)):
            parts = []
            for item in value:
                if isinstance(item, torch.Tensor):
                    parts.append(item.detach().cpu().numpy())
                else:
                    parts.append(np.asarray(item))
            if not parts:
                return None
            arr = np.concatenate([p.reshape(-1) if p.ndim == 0 else p for p in parts], axis=0)
        else:
            arr = np.asarray(value)
    except Exception:
        return None
    if arr.size == 0:
        return None
    if score and arr.ndim == 2:
        if arr.shape[1] == 1:
            arr = arr[:, 0]
        elif arr.shape[1] >= 2:
            arr = arr[:, 1]
        else:
            arr = arr.reshape(-1)
    elif arr.ndim > 1:
        if arr.shape[-1] == 1:
            arr = arr.reshape(-1)
        else:
            # For labels/predictions this normally means one-hot or class-probabilities.
            # Use argmax when not explicitly a score.
            arr = arr.argmax(axis=1)
    return arr.reshape(-1)


def _from_stats_array(key):
    stats_obj = globals().get("stats")
    if isinstance(stats_obj, dict) and key in stats_obj:
        return _as_array(stats_obj.get(key), score=(key in ["y_score", "score", "proba", "prob"]))
    return None


def _find_array(names, *, target_len=None, score=False):
    # First check arrays stored inside stats returned by the notebook evaluation.
    stats_key_map = {
        "true": ["y_true", "true_label", "labels", "y"],
        "pred": ["y_pred", "pred_label", "pred", "prediction"],
        "score": ["y_score", "score", "proba", "prob", "y_prob"],
    }
    key_group = "score" if score else None
    if not score and any("pred" in n for n in names):
        key_group = "pred"
    if not score and any("true" in n or n in ["y", "ytrue"] for n in names):
        key_group = "true"
    if key_group:
        for k in stats_key_map[key_group]:
            arr = _from_stats_array(k)
            if arr is not None and (target_len is None or len(arr) == target_len):
                return arr

    for name in names:
        if name not in globals():
            continue
        arr = _as_array(globals().get(name), score=score)
        if arr is None:
            continue
        if target_len is not None and len(arr) != target_len:
            continue
        return arr
    return None


def _scalar_from_stats(*keys):
    stats_obj = globals().get("stats")
    if isinstance(stats_obj, dict):
        for key in keys:
            if key in stats_obj and _is_number(stats_obj[key]):
                return float(stats_obj[key])
    return float("nan")


def _scalar_from_globals(names):
    for name in names:
        if name in globals() and _is_number(globals()[name]):
            return float(globals()[name])
    return float("nan")


def _collect_epoch_metrics():
    if isinstance(globals().get("hist"), dict):
        h = globals()["hist"]
        key_map = {
            "tr_loss": "train_loss", "va_loss": "val_loss",
            "tr_acc": "train_accuracy", "va_acc": "val_accuracy",
            "tr_auc": "train_roc_auc", "va_auc": "val_roc_auc",
            "tr_pra": "train_pr_auc", "va_pra": "val_pr_auc",
            "thr_tr": "train_throughput_samples_per_sec",
            "thr_va": "val_throughput_samples_per_sec",
            "cpu": "cpu_percent", "ram": "ram_used_GB", "gpu_peak": "gpu_peak_MB",
        }
        rows = {out_name: list(h[in_name]) for in_name, out_name in key_map.items() if in_name in h and len(h[in_name]) > 0}
    else:
        candidates = {
            "train_loss": ["tr_loss_hist", "train_losses"],
            "val_loss": ["va_loss_hist", "val_losses", "test_losses"],
            "train_accuracy": ["tr_acc_hist", "train_accuracies"],
            "val_accuracy": ["va_acc_hist", "val_accuracies", "test_accuracies"],
            "train_roc_auc": ["tr_auc_hist"],
            "val_roc_auc": ["va_auc_hist", "test_auc_hist"],
            "train_pr_auc": ["tr_pra_hist"],
            "val_pr_auc": ["va_pra_hist", "val_praucs", "test_prauc_hist"],
            "val_f1": ["val_f1s", "test_f1_hist"],
            "train_throughput_samples_per_sec": ["thr_tr_hist", "train_throughputs", "thr_train"],
            "val_throughput_samples_per_sec": ["thr_va_hist", "val_throughputs", "test_throughputs", "thr_val"],
            "cpu_percent": ["cpu_hist", "cpu_list"],
            "ram_used_GB": ["ram_hist", "ram_list"],
            "gpu_peak_MB": ["gpu_peak_hist", "gpu_peak_list"],
            "epoch_time_sec": ["epoch_times"],
        }
        rows = {}
        for out_name, names in candidates.items():
            for name in names:
                value = globals().get(name)
                if isinstance(value, (list, tuple, np.ndarray)) and len(value) > 0:
                    rows[out_name] = list(value)
                    break
    if not rows:
        return pd.DataFrame()
    max_len = max(len(v) for v in rows.values())
    data = {"epoch": list(range(1, max_len + 1))}
    for key, values in rows.items():
        values = list(values)
        data[key] = values + [np.nan] * (max_len - len(values))
    return pd.DataFrame(data)


def _plot_confusion_matrix(cm_arr, path):
    if cm_arr is None:
        return
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_arr)
    ax.set_title("Confusion Matrix")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["0", "1"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["0", "1"])
    for i in range(cm_arr.shape[0]):
        for j in range(cm_arr.shape[1]):
            ax.text(j, i, str(cm_arr[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def _plot_roc_curve(y_true_arr, y_score_arr, path):
    if y_true_arr is None or y_score_arr is None or len(np.unique(y_true_arr)) != 2:
        return pd.DataFrame()
    fpr_arr, tpr_arr, thresholds = roc_curve(y_true_arr, y_score_arr)
    auc_value = roc_auc_score(y_true_arr, y_score_arr)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr_arr, tpr_arr, label=f"ROC-AUC = {auc_value:.4f}")
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return pd.DataFrame({"fpr": fpr_arr, "tpr": tpr_arr, "threshold": thresholds})


def _plot_pr_curve(y_true_arr, y_score_arr, path):
    if y_true_arr is None or y_score_arr is None or len(np.unique(y_true_arr)) != 2:
        return pd.DataFrame()
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true_arr, y_score_arr)
    pr_auc_value = average_precision_score(y_true_arr, y_score_arr)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall_arr, precision_arr, label=f"PR-AUC = {pr_auc_value:.4f}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.legend(loc="lower left")
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    thresholds_padded = np.append(thresholds, np.nan)
    return pd.DataFrame({"recall": recall_arr, "precision": precision_arr, "threshold": thresholds_padded})


def _plot_epoch_curves(epoch_df, path):
    if epoch_df.empty:
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    if "train_loss" in epoch_df:
        ax.plot(epoch_df["epoch"], epoch_df["train_loss"], label="Train loss")
    if "val_loss" in epoch_df:
        ax.plot(epoch_df["epoch"], epoch_df["val_loss"], label="Validation/Test loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Loss Curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path / "loss_curve.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    if "train_accuracy" in epoch_df:
        ax.plot(epoch_df["epoch"], epoch_df["train_accuracy"], label="Train accuracy")
    if "val_accuracy" in epoch_df:
        ax.plot(epoch_df["epoch"], epoch_df["val_accuracy"], label="Validation/Test accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy Curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(path / "accuracy_curve.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    plotted = False
    for col, label in [
        ("val_roc_auc", "Validation/Test ROC-AUC"),
        ("val_pr_auc", "Validation/Test PR-AUC"),
        ("val_f1", "Validation/Test F1"),
    ]:
        if col in epoch_df:
            ax.plot(epoch_df["epoch"], epoch_df[col], label=label)
            plotted = True
    if plotted:
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Score")
        ax.set_title("Validation/Test Score Curves")
        ax.legend()
        fig.tight_layout()
        fig.savefig(path / "score_curves.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# -------------------- Collect existing evaluation output --------------------
model_name, final_model = _find_model()
software_name = str(globals().get("software", "unknown_software"))
notebook_stem = Path(NOTEBOOK_NAME).stem if NOTEBOOK_NAME != "unknown_notebook.ipynb" else "unknown_notebook"
batch_id = str(globals().get("BATCH_ID", "NA"))
execution_number = globals().get("EXECUTION_NUMBER", None)
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", r"E:\RECUPERACAO_HD\arm_arc - Paper ICC\CNSM_2026"))
NOTEBOOKS_DIR = Path(globals().get("NOTEBOOKS_DIR", PROJECT_ROOT / "notebooks"))
RESULTS_ROOT = Path(globals().get("RESULTS_ROOT", PROJECT_ROOT / "results"))
SHARED_ARTIFACTS_ROOT = Path(globals().get("SHARED_ARTIFACTS_ROOT", PROJECT_ROOT / "shared_artifacts"))
DADO_PICKLE_ROOT = Path(globals().get("DADO_PICKLE_ROOT", PROJECT_ROOT / "dado_pickle"))
run_id = str(globals().get("RUN_ID", f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{notebook_stem}_{software_name}_{uuid.uuid4().hex[:8]}"))
run_dir = Path(globals().get("RUN_DIR", RESULTS_ROOT / "runs" / run_id))

dataset_id = str(globals().get("DATASET_ID", "NA"))
dataset_manifest_path = str(globals().get("DATASET_MANIFEST_PATH", ""))
processing_run_dir = str(globals().get("PROCESSING_RUN_DIR", ""))
processed_data_root = str(globals().get("PROCESSED_DATA_ROOT", ""))

fig_dir = run_dir / "figures"
model_dir = run_dir / "model"
run_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

# Prefer arrays already created by the notebook evaluation.
y_true = _find_array(["ytrue", "y_true_all", "y_true", "all_targets", "all_labels", "y_test"], score=False)
y_score = _find_array(["yprob", "y_prob_all", "proba", "y_score", "y_scores", "all_probs", "all_scores"], target_len=(len(y_true) if y_true is not None else None), score=True)
y_pred = _find_array(["y_pred_test", "ypred", "y_pred", "pred", "all_preds", "preds"], target_len=(len(y_true) if y_true is not None else None), score=False)

if y_pred is None and y_score is not None:
    y_pred = (y_score >= 0.5).astype(int)
if y_true is not None:
    y_true = y_true.astype(int)
if y_pred is not None:
    y_pred = y_pred.astype(int)

# Confusion matrix from notebook output or from collected predictions.
cm_existing = None
if isinstance(globals().get("stats"), dict) and "cm" in globals()["stats"]:
    cm_existing = np.asarray(globals()["stats"]["cm"])
elif "cm" in globals():
    try:
        cm_existing = np.asarray(globals()["cm"])
    except Exception:
        cm_existing = None
if cm_existing is not None and cm_existing.shape == (2, 2):
    cm = cm_existing.astype(int)
elif y_true is not None and y_pred is not None:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
else:
    cm = None

# Metrics from stats dict or existing scalar variables. Compute missing values when arrays exist.
metrics = {
    "accuracy": _scalar_from_stats("accuracy", "acc", "test_acc"),
    "precision": _scalar_from_stats("precision", "prec"),
    "recall": _scalar_from_stats("recall", "rec"),
    "f1_score": _scalar_from_stats("f1_score", "f1"),
    "loss": _scalar_from_stats("loss", "test_loss"),
    "roc_auc": _scalar_from_stats("roc_auc", "auc", "auc_score"),
    "pr_auc": _scalar_from_stats("pr_auc", "prauc", "pr"),
    "throughput_samples_per_sec": _scalar_from_stats("throughput_samples_per_sec", "throughput", "thr"),
    "gpu_peak_MB": _scalar_from_stats("gpu_peak_MB", "gpu_peak", "gpk"),
    "elapsed_sec": _scalar_from_stats("elapsed_sec", "test_time", "dt"),
}
metrics["accuracy"] = metrics["accuracy"] if _is_number(metrics["accuracy"]) else _scalar_from_globals(["test_acc", "accuracy", "acc"])
metrics["precision"] = metrics["precision"] if _is_number(metrics["precision"]) else _scalar_from_globals(["test_precision", "precision", "prec"])
metrics["recall"] = metrics["recall"] if _is_number(metrics["recall"]) else _scalar_from_globals(["test_recall", "recall", "rec"])
metrics["f1_score"] = metrics["f1_score"] if _is_number(metrics["f1_score"]) else _scalar_from_globals(["f1_test", "f1"])
metrics["loss"] = metrics["loss"] if _is_number(metrics["loss"]) else _scalar_from_globals(["test_loss"])
metrics["roc_auc"] = metrics["roc_auc"] if _is_number(metrics["roc_auc"]) else _scalar_from_globals(["roc_auc", "auc_score", "auc"])
metrics["pr_auc"] = metrics["pr_auc"] if _is_number(metrics["pr_auc"]) else _scalar_from_globals(["pr_auc_test", "pr_auc"])
metrics["throughput_samples_per_sec"] = metrics["throughput_samples_per_sec"] if _is_number(metrics["throughput_samples_per_sec"]) else _scalar_from_globals(["test_thr", "thr_test", "throughput", "thr"])
metrics["gpu_peak_MB"] = metrics["gpu_peak_MB"] if _is_number(metrics["gpu_peak_MB"]) else _scalar_from_globals(["gpu_pk_test", "gpu_peak_test", "gpk", "gpu_peak_MB"])
metrics["elapsed_sec"] = metrics["elapsed_sec"] if _is_number(metrics["elapsed_sec"]) else _scalar_from_globals(["test_time"])

if y_true is not None and y_pred is not None and len(y_true) == len(y_pred):
    if not _is_number(metrics["accuracy"]):
        metrics["accuracy"] = accuracy_score(y_true, y_pred)
    if not _is_number(metrics["precision"]):
        metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
    if not _is_number(metrics["recall"]):
        metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
    if not _is_number(metrics["f1_score"]):
        metrics["f1_score"] = f1_score(y_true, y_pred, zero_division=0)
if y_true is not None and y_score is not None and len(y_true) == len(y_score) and len(np.unique(y_true)) == 2:
    if not _is_number(metrics["roc_auc"]):
        metrics["roc_auc"] = roc_auc_score(y_true, y_score)
    if not _is_number(metrics["pr_auc"]):
        metrics["pr_auc"] = average_precision_score(y_true, y_score)

if cm is not None:
    tn, fp, fn, tp = cm.ravel()
else:
    tn = fp = fn = tp = np.nan
metrics.update({
    "tn": int(tn) if _is_number(tn) else np.nan,
    "fp": int(fp) if _is_number(fp) else np.nan,
    "fn": int(fn) if _is_number(fn) else np.nan,
    "tp": int(tp) if _is_number(tp) else np.nan,
    "fpr": (fp / max(1, fp + tn)) if _is_number(fp) and _is_number(tn) else np.nan,
    "fnr": (fn / max(1, fn + tp)) if _is_number(fn) and _is_number(tp) else np.nan,
    "n_test_samples": int(len(y_true)) if y_true is not None else int(_scalar_from_globals(["n_test", "test_total"])) if _is_number(_scalar_from_globals(["n_test", "test_total"])) else np.nan,
})
metrics["Throughput"] = metrics["throughput_samples_per_sec"]
metrics["thouput"] = metrics["throughput_samples_per_sec"]

# Model metadata and checkpoint. This does not re-run inference.
if final_model is not None:
    checkpoint_path = model_dir / "checkpoint_state_dict.pth"
    torch.save({
        "state_dict": final_model.state_dict(),
        "model_class": final_model.__class__.__name__,
        "model_variable_name": model_name,
        "notebook_name": NOTEBOOK_NAME,
        "software": software_name,
        "tracking_version": STANDARD_TRACKING_VERSION,
        "run_id": run_id,
        "batch_id": batch_id,
        "execution_number": execution_number,
        "dataset_id": dataset_id,
        "dataset_manifest_path": dataset_manifest_path,
        "processing_run_dir": processing_run_dir,
    }, checkpoint_path)
    n_params_value = _scalar_from_globals(["n_params"])
    if not _is_number(n_params_value):
        n_params_value = sum(p.numel() for p in final_model.parameters())
    trainable_params_value = sum(p.numel() for p in final_model.parameters() if p.requires_grad)
    model_size_value = _scalar_from_globals(["size_mb", "model_size_MB"])
    if not _is_number(model_size_value):
        model_size_value = checkpoint_path.stat().st_size / (1024 ** 2)
    model_class_value = final_model.__class__.__name__
    device_value = str(_get_device(final_model))
else:
    checkpoint_path = None
    n_params_value = _scalar_from_globals(["n_params"])
    trainable_params_value = np.nan
    model_size_value = _scalar_from_globals(["size_mb", "model_size_MB"])
    model_class_value = "unknown"
    device_value = str(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

macs_per_sample = _scalar_from_globals(["macs_per_sample", "n_macs", "macs", "MACs"])
macs_note = "from_existing_notebook_variable" if _is_number(macs_per_sample) else "not_available_without_extra_forward_pass_existing_evaluation_only"

metrics.update({
    "run_id": run_id,
    "batch_id": batch_id,
    "execution_number": execution_number,
    "dataset_id": dataset_id,
    "dataset_manifest_path": dataset_manifest_path,
    "processing_run_dir": processing_run_dir,
    "processed_data_root": processed_data_root,
    "date_time": datetime.now().isoformat(timespec="seconds"),
    "notebook_name": NOTEBOOK_NAME,
    "software": software_name,
    "model_variable_name": model_name,
    "model_class": model_class_value,
    "n_params": int(n_params_value) if _is_number(n_params_value) else np.nan,
    "trainable_params": int(trainable_params_value) if _is_number(trainable_params_value) else np.nan,
    "macs_per_sample": float(macs_per_sample) if _is_number(macs_per_sample) else np.nan,
    "macs_estimation_note": macs_note,
    "model_size_MB": float(model_size_value) if _is_number(model_size_value) else np.nan,
    "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else None,
    "run_dir": str(run_dir),
    "tracking_version": STANDARD_TRACKING_VERSION,
    "tracking_note": "Saved from metrics already computed by the notebook; no additional test-loader evaluation was executed.",
})

# Final metrics table.
summary_columns = [
    "date_time", "batch_id", "execution_number", "run_id", "dataset_id", "notebook_name", "software", "model_class",
    "accuracy", "precision", "recall", "f1_score", "loss", "roc_auc", "pr_auc",
    "tn", "fp", "fn", "tp", "fpr", "fnr",
    "n_params", "trainable_params", "macs_per_sample", "model_size_MB",
    "throughput_samples_per_sec", "Throughput", "thouput", "gpu_peak_MB",
    "elapsed_sec", "n_test_samples", "checkpoint_path", "run_dir", "macs_estimation_note", "tracking_note",
]
summary_df = pd.DataFrame([{k: metrics.get(k, np.nan) for k in summary_columns}])
summary_df.to_csv(run_dir / "test_metrics_summary.csv", index=False)

# Predictions table, when the notebook exposes predictions.
if y_true is not None and y_pred is not None and len(y_true) == len(y_pred):
    idx_test_values = globals().get("idx_test", None)
    if idx_test_values is not None and len(idx_test_values) == len(y_true):
        original_index = np.asarray(idx_test_values)
    else:
        original_index = np.arange(len(y_true))
    outcome = np.where(
        (y_true == 1) & (y_pred == 1), "TP",
        np.where((y_true == 0) & (y_pred == 0), "TN", np.where(y_pred == 1, "FP", "FN"))
    )
    pred_df = pd.DataFrame({
        "sample_index": np.arange(len(y_true)),
        "original_index": original_index,
        "true_label": y_true,
        "pred_label": y_pred,
        "pred_score": y_score if y_score is not None and len(y_score) == len(y_true) else np.nan,
        "outcome": outcome,
    })
    pred_df.to_csv(run_dir / "predictions_test.csv", index=False)
else:
    pred_df = pd.DataFrame()

# Confusion matrix table.
if cm is not None:
    pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(run_dir / "confusion_matrix.csv")

# Epoch metrics table.
epoch_df = _collect_epoch_metrics()
if not epoch_df.empty:
    epoch_df.to_csv(run_dir / "metrics_epoch.csv", index=False)

# Curve points and figures.
if cm is not None:
    _plot_confusion_matrix(cm, fig_dir / "confusion_matrix.png")
roc_points = _plot_roc_curve(y_true, y_score, fig_dir / "roc_auc_curve.png")
if not roc_points.empty:
    roc_points.to_csv(run_dir / "roc_curve_points.csv", index=False)
pr_points = _plot_pr_curve(y_true, y_score, fig_dir / "pr_auc_curve.png")
if not pr_points.empty:
    pr_points.to_csv(run_dir / "pr_curve_points.csv", index=False)
_plot_epoch_curves(epoch_df, fig_dir)

# Config/environment metadata.
config = {
    "notebook_name": NOTEBOOK_NAME,
    "software": software_name,
    "batch_id": batch_id,
    "execution_number": execution_number,
    "run_id": run_id,
    "project_root": str(PROJECT_ROOT),
    "results_root": str(RESULTS_ROOT),
    "shared_artifacts_root": str(SHARED_ARTIFACTS_ROOT),
    "dado_pickle_root": str(DADO_PICKLE_ROOT),
    "dataset_id": dataset_id,
    "dataset_manifest_path": dataset_manifest_path,
    "processing_run_dir": processing_run_dir,
    "processed_data_root": processed_data_root,
    "model_class": model_class_value,
    "model_variable_name": model_name,
    "device": device_value,
    "batch_size": globals().get("batch_size", globals().get("BATCH_SIZE", None)),
    "num_epochs": globals().get("num_epochs", globals().get("EPOCHS", globals().get("epochs", None))),
    "learning_rate": globals().get("LR", globals().get("lr", None)),
    "seed": globals().get("SEED", None),
    "tracking_version": STANDARD_TRACKING_VERSION,
}
with open(run_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump(_json_safe(config), f, indent=2)

environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
with open(run_dir / "environment.json", "w", encoding="utf-8") as f:
    json.dump(_json_safe(environment), f, indent=2)

artifacts = {
    "batch_id": batch_id,
    "execution_number": execution_number,
    "run_id": run_id,
    "dataset_id": dataset_id,
    "dataset_manifest_path": dataset_manifest_path,
    "processing_run_dir": processing_run_dir,
    "test_metrics_summary": str(run_dir / "test_metrics_summary.csv"),
    "predictions_test": str(run_dir / "predictions_test.csv") if not pred_df.empty else None,
    "confusion_matrix_csv": str(run_dir / "confusion_matrix.csv") if cm is not None else None,
    "metrics_epoch": str(run_dir / "metrics_epoch.csv") if not epoch_df.empty else None,
    "checkpoint": str(checkpoint_path) if checkpoint_path is not None else None,
    "figures_dir": str(fig_dir),
    "roc_curve_points": str(run_dir / "roc_curve_points.csv") if not roc_points.empty else None,
    "pr_curve_points": str(run_dir / "pr_curve_points.csv") if not pr_points.empty else None,
}
with open(run_dir / "artifacts.json", "w", encoding="utf-8") as f:
    json.dump(_json_safe(artifacts), f, indent=2)
with open(run_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(_json_safe(metrics), f, indent=2)

# Global cumulative summary for quick consultation.
summary_root = RESULTS_ROOT
summary_root.mkdir(parents=True, exist_ok=True)
global_summary_path = summary_root / "results_summary_model.csv"
if global_summary_path.exists():
    summary_df.to_csv(global_summary_path, mode="a", header=False, index=False)
else:
    summary_df.to_csv(global_summary_path, index=False)

print("\n=== Standardized scientific tracking completed ===")
print("Mode: existing notebook evaluation only; no additional test-loader evaluation was executed.")
print(f"Run ID: {run_id}")
print(f"Run directory: {run_dir}")
print(summary_df.T.rename(columns={0: "value"}))